In [2]:
import pandas as pd
import numpy as np
import logging

**CONFIGURE LOGGING**

In [4]:
logging.basicConfig(
    filename='../logs/cleaning.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("Pipeline started")

**LOAD DATASET**

In [5]:
logging.info("Loading dataset...")

In [7]:
df = pd.read_csv(
    "../data_row/attacks.csv",
    encoding="latin1"
)

In [8]:
logging.info(f"Dataset loaded successfully with shape {df.shape}")


In [9]:
df.head()

,Case Number,Date,Year,Type,Country,Area,Location,Activity,Name,Sex,...,Fatal (Y/N),Time,Species,Investigator or Source,pdf,href formula,href,Case Number.1,Case Number.2,original order
0,2017.06.11,2017-06-11,2017.0,Unprovoked,AUSTRALIA,Western Australia,"Point Casuarina, Bunbury",Body boarding,Paul Goff,M,...,N,08h30,"White shark, 4 m","WA Today, 6/11/2017",2017.06.11-Goff.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.11,2017.06.11,6095.0
1,2017.06.10.b,2017-06-10,2017.0,Unprovoked,AUSTRALIA,Victoria,"Flinders, Mornington Penisula",Surfing,female,F,...,N,15h45,7 gill shark,NaN,2017.06.10.b-Flinders.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.10.b,2017.06.10.b,6094.0
2,2017.06.10.a,2017-06-10,2017.0,Unprovoked,USA,Florida,"Ponce Inlet, Volusia County",Surfing,Bryan Brock,M,...,N,10h00,NaN,"Daytona Beach News-Journal, 6/10/2017",2017.06.10.a-Brock.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.10.a,2017.06.10.a,6093.0
3,2017.06.07.R,Reported 07-Jun-2017,2017.0,Unprovoked,UNITED KINGDOM,South Devon,Bantham Beach,Surfing,Rich Thomson,M,...,N,NaN,"3m shark, probably a smooth hound","C. Moore, GSAF",2017.06.07.R-Thomson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.07.R,2017.06.07.R,6092.0
4,2017.06.04,2017-06-04,2017.0,Unprovoked,USA,Florida,"Middle Sambo Reef off Boca Chica, Monroe County",Spearfishing,Parker Simpson,M,...,N,NaN,8' shark,"Nine News, 6/7/2017",2017.06.04-Simpson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.04,2017.06.04,6091.0


In [10]:
assert df.shape[0] > 0, "Dataset contains no rows!"
assert df.shape[1] > 0, "Dataset contains no columns!"

In [11]:
logging.info("Initial dataset validation passed")

In [12]:
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(25614, 22)


In [13]:
print("Column Names:")
print(df.columns.tolist())

Column Names:
['Case Number', 'Date', 'Year', 'Type', 'Country', 'Area', 'Location', 'Activity', 'Name', 'Sex ', 'Age', 'Injury', 'Fatal (Y/N)', 'Time', 'Species ', 'Investigator or Source', 'pdf', 'href formula', 'href', 'Case Number.1', 'Case Number.2', 'original order']


In [14]:
logging.info("Cleaning column names")

In [15]:
df.columns = df.columns.str.strip()

In [16]:
logging.info("Column names cleaned successfully")

In [17]:
print(df.columns.tolist())

['Case Number', 'Date', 'Year', 'Type', 'Country', 'Area', 'Location', 'Activity', 'Name', 'Sex', 'Age', 'Injury', 'Fatal (Y/N)', 'Time', 'Species', 'Investigator or Source', 'pdf', 'href formula', 'href', 'Case Number.1', 'Case Number.2', 'original order']


In [18]:
print(df.dtypes)

Case Number                   str
Date                          str
Year                      float64
Type                          str
Country                       str
Area                          str
Location                      str
Activity                      str
Name                          str
Sex                           str
Age                           str
Injury                        str
Fatal (Y/N)                   str
Time                          str
Species                       str
Investigator or Source        str
pdf                           str
href formula                  str
href                          str
Case Number.1                 str
Case Number.2                 str
original order            float64
dtype: object


In [19]:
logging.info("Removing fully blank rows")

In [20]:
initial_rows = df.shape[0]

In [21]:
df = df.dropna(how='all')


In [22]:
final_rows = df.shape[0]

In [23]:
removed_rows = initial_rows - final_rows

In [24]:
logging.info(f"Removed {removed_rows} blank rows")

In [25]:
print(f"Removed Rows: {removed_rows}")
print(f"New Shape: {df.shape}")

Removed Rows: 19518
New Shape: (6096, 22)


In [26]:
missing_values = df.isnull().sum()
missing_percent = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

In [28]:
print("Missing Values Percentage:")
print(missing_percent)

Missing Values Percentage:
Time                      53.280840
Species                   49.146982
Age                       44.652231
Sex                        9.498031
Activity                   8.809055
Location                   8.415354
Area                       6.791339
Name                       3.412073
Country                    0.787402
Fatal (Y/N)                0.524934
Injury                     0.492126
Investigator or Source     0.311680
Type                       0.098425
Year                       0.065617
href formula               0.049213
href                       0.049213
pdf                        0.032808
Date                       0.032808
Case Number.2              0.032808
Case Number.1              0.032808
original order             0.032808
Case Number                0.016404
dtype: float64 %


In [29]:
print("Unique values in Fatal (Y/N):")
print(df['Fatal (Y/N)'].unique())

Unique values in Fatal (Y/N):
<StringArray>
['N', 'Y', nan, 'UNKNOWN', '2017', ' N', 'F', 'N ', '#VALUE!', 'n']
Length: 10, dtype: str


In [30]:
logging.info("Cleaning Fatal (Y/N) column")

In [31]:
df['Fatal (Y/N)'] = (
    df['Fatal (Y/N)']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [32]:
print(df['Fatal (Y/N)'].unique())

<StringArray>
['N', 'Y', nan, 'UNKNOWN', '2017', 'F', '#VALUE!']
Length: 7, dtype: str
